# SetScout — prototype notebook

Thin driver for the `setscout` package (`setscout/graph/`, `models/`, `tools/`, `pipeline.py`).

**Setup:** `conda env create -f environment.yml && conda activate setscout && pip install -e .`

Repo-root `.env` keys: `GEMINI_API_KEY`, `KAGGLE_USERNAME`, `KAGGLE_KEY`.

In [ ]:
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display

from setscout import build_setscout_graph, run_pipeline

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(REPO_ROOT / ".env")

In [ ]:
try:
    display(Markdown("```mermaid\n" + build_setscout_graph(None).get_graph().draw_mermaid() + "\n```"))
except Exception as e:
    print("Graph viz skipped:", e)

## User request

Edit `REQUEST`, then run the pipeline cell.

In [ ]:
REQUEST = {
    # --- mandatory ---
    "purpose": "Train a multilingual sentiment classifier to route negative apparel reviews to customer support and flag sizing/quality complaints automatically",
    "domain": "e-commerce / fashion retail NLP",
    "data_type": "text reviews with star ratings (1–5) and product category labels",
    # --- optional ---
    "requirements": "Min 50k labeled reviews; at least 3 languages (English, Spanish, French); per-review star rating + free-text body; public or CC-BY license; reviews from 2018 onward; avoid synthetic/LLM-generated text; max avg class imbalance 80/20 for positive vs negative",
    "additional_notes": "Prefer datasets with product category metadata (e.g. shoes, dresses). Bonus if size/fit mentions are common in negative reviews.",
    "exclude_datasets": ["amazon-reviews-2023", "yelp-open-dataset"],
}

In [ ]:
result = run_pipeline(REQUEST)

print("--- pipeline logs ---")
for line in result.get("logs", []):
    print(line)
print("\n--- search_spec ---")
print(result["search_spec"].model_dump_json(indent=2))
print("\n--- candidates ---")
for c in result.get("candidates", []):
    print(c.model_dump_json(indent=2))
display(Markdown(result["report"]))

## Inspect final state (optional)

In [ ]:
# import json
# json.loads(json.dumps({k: v for k, v in result.items() if k != "report"}, default=str))